In [1]:
import pandas as pd
import numpy as np

BASE = '../../new-dataset/tennis_MatchChartingProject-master'

matches           = pd.read_csv(f'{BASE}/charting-m-matches.csv')
feat_overview     = pd.read_csv(f'{BASE}/features_overview.csv')
feat_keypoints    = pd.read_csv(f'{BASE}/features_keypoints.csv')
feat_rally        = pd.read_csv(f'{BASE}/features_rally.csv')
winners           = pd.read_csv(f'{BASE}/winners.csv')

matches['Date'] = pd.to_datetime(matches['Date'], format='%Y%m%d', errors='coerce')
feat_overview['Date']  = pd.to_datetime(feat_overview['Date'],  errors='coerce')
feat_keypoints['Date'] = pd.to_datetime(feat_keypoints['Date'], errors='coerce')
feat_rally['Date']     = pd.to_datetime(feat_rally['Date'],     errors='coerce')

print('matches:        ', matches.shape)
print('feat_overview:  ', feat_overview.shape)
print('feat_keypoints: ', feat_keypoints.shape)
print('feat_rally:     ', feat_rally.shape)

matches:         (7566, 15)
feat_overview:   (15116, 13)
feat_keypoints:  (15090, 7)
feat_rally:      (15090, 9)


In [2]:
matches = matches.merge(winners, how='left')
matches.sample(5)

,match_id,Player 1,Player 2,Pl 1 hand,Pl 2 hand,Date,Tournament,Round,Time,Court,Surface,Umpire,Best of,Final TB?,Charted by,Winner
7455,19850818-M-Canada_Masters-F-John_Mcenroe-Ivan_...,John Mcenroe,Ivan Lendl,L,R,1985-08-18,Canada Masters,F,3pm,Centre,Hard,Jeremy Shales,3,1,Edo,1.0
332,20250723-M-Washington-R32-Reilly_Opelka-Daniil...,Reilly Opelka,Daniil Medvedev,R,R,2025-07-23,Washington,R32,NaN,NaN,Hard,NaN,3,1,Ludo,2.0
6284,20030628-M-Wimbledon-R32-Andre_Agassi-Younes_E...,Andre Agassi,Younes El Aynaoui,R,R,2003-06-28,Wimbledon,R32,1:10 PM,Centre,Grass,NaN,5,0,BG,1.0
977,20240815-M-Cincinnati_Masters-R32-Gael_Monfils...,Gael Monfils,Carlos Alcaraz,R,R,2024-08-15,Cincinnati Masters,R32,19:10,Center,Hard,NaN,3,1,Zindaras,1.0
5648,20090815-M-Canada_Masters-SF-Andy_Murray-Jo_Wi...,Andy Murray,Jo Wilfried Tsonga,R,R,2009-08-15,Canada Masters,SF,2pm,Centre,Hard,Steve Ullrich,3,1,Edo,1.0


In [3]:
player_profiles = (feat_overview
    .merge(feat_keypoints, on=['match_id', 'player', 'Date'], how='left')
    .merge(feat_rally,     on=['match_id', 'player', 'Date'], how='left')
    .merge(winners, on ='match_id', how='left')
    )

player_profiles = player_profiles.drop_duplicates(['match_id', 'player'])
print(player_profiles.shape)

(15090, 24)


In [4]:
player_profiles.head()

,match_id,player,Date,avg_first_serve_pct,avg_first_serve_won_pct,avg_second_serve_won_pct,avg_ace_pct,avg_df_pct,avg_return_won_pct,avg_winners_per_pt,...,avg_bp_first_in_pct,avg_bpo_conv_pct,avg_bpo_ue_pct,avg_srv_win_1_3,avg_srv_win_4_6,avg_srv_win_7plus,avg_ret_win_1_3,avg_ret_win_4_6,avg_ret_win_7plus,Winner
0,19890909-M-US_Open-SF-Boris_Becker-Aaron_Krick...,Aaron Krickstein,1989-09-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,19900415-M-Tokyo_Outdoor-F-Aaron_Krickstein-St...,Aaron Krickstein,1990-04-15,0.447917,0.720930,0.377358,0.062500,0.041667,0.414414,0.149758,...,0.452381,0.345455,0.109091,NaN,NaN,NaN,0.447619,0.44898,0.528302,2.0
2,19910902-M-US_Open-R16-Aaron_Krickstein-Jimmy_...,Aaron Krickstein,1991-09-02,0.576522,0.669556,0.362592,0.050481,0.020833,0.401652,0.151546,...,0.554762,0.382405,0.086804,0.439024,0.454545,0.542857,0.447619,0.44898,0.528302,2.0
3,19911026-M-Stockholm_Masters-SF-Aaron_Krickste...,Aaron Krickstein,1991-10-26,0.570086,0.666609,0.432739,0.045262,0.017206,0.396423,0.133289,...,0.562281,0.401995,0.087281,0.503200,0.425189,0.541799,0.447619,0.44898,0.528302,2.0
4,19920426-M-Monte_Carlo_Masters-F-Aaron_Krickst...,Aaron Krickstein,1992-04-26,0.570422,0.633885,0.419793,0.054355,0.012904,0.363984,0.123903,...,0.546711,0.375026,0.080166,0.473398,0.341431,0.540686,0.447619,0.44898,0.528302,2.0


Fazendo a função para calcular rating

In [5]:
#probabilidade de a ganhar de b
def expected_score(r_a, r_b):
    return 1/(1+10**((r_b - r_a)/400))

#pontos de rating a mais
def calc_new_rating(r_a, r_b, win):
    k = 32
    return r_a + k*(win - expected_score(r_a, r_b))

In [6]:
from collections import defaultdict

def compute_rating_history(matches, k=32, initial=1500):
    ratings = defaultdict(lambda: initial)
    ratings_surface = defaultdict(lambda: defaultdict(lambda: initial))
    records = []

    for _, row in matches.sort_values('Date').iterrows():
        p1, p2 = row['Player 1'], row['Player 2']
        r1, r2 = ratings[p1], ratings[p2]
        surface = row['Surface'] if pd.notna(row['Surface']) else None
        rs1 = ratings_surface[surface][p1] if surface else None
        rs2 = ratings_surface[surface][p2] if surface else None

        records.append({'match_id': row['match_id'], 'player': p1, 'rating_before': r1, 'rating_surface_before': rs1})
        records.append({'match_id': row['match_id'], 'player': p2, 'rating_before': r2, 'rating_surface_before': rs2})

        if pd.notna(row['Winner']):
            p1_won = int(row['Winner'] == 1.0)
            ratings[p1] = calc_new_rating(r1, r2, p1_won)
            ratings[p2] = calc_new_rating(r2, r1, 1 - p1_won)
            if surface:
                ratings_surface[surface][p1] = calc_new_rating(rs1, rs2, p1_won)
                ratings_surface[surface][p2] = calc_new_rating(rs2, rs1, 1 - p1_won)

    return pd.DataFrame(records)

rating_df = compute_rating_history(matches)
player_profiles = player_profiles.merge(rating_df, on=['match_id', 'player'], how='left')
player_profiles[['player', 'rating_before', 'rating_surface_before']].head(10)

,player,rating_before,rating_surface_before
0,Aaron Krickstein,1500.000000,1500.000000
1,Aaron Krickstein,1490.400904,1487.859408
2,Aaron Krickstein,1479.469917,1475.175244
3,Aaron Krickstein,1464.071819,1461.084075
4,Aaron Krickstein,1458.673966,1500.000000
5,Aaron Krickstein,1441.744624,1455.172019
6,Aaron Krickstein,1427.632292,1440.201258
7,Aaron Krickstein,1450.069110,1463.472684
8,Abedallah Shelbayh,1500.000000,1500.000000
9,Abedallah Shelbayh,1484.323476,1483.882257


In [7]:
player_profiles = player_profiles.merge(matches[['match_id', 'Surface']], on='match_id', how='left')

In [8]:
hand_p1 = matches[['Player 1', 'Pl 1 hand']].rename(columns={'Player 1': 'player', 'Pl 1 hand': 'hand'})
hand_p2 = matches[['Player 2', 'Pl 2 hand']].rename(columns={'Player 2': 'player', 'Pl 2 hand': 'hand'})

player_hand = (pd.concat([hand_p1, hand_p2])
    .dropna(subset=['hand'])
    .query("hand in ['L', 'R']")
    .drop_duplicates('player')
)

player_profiles = player_profiles.merge(player_hand, on='player', how='left')
player_profiles[['player', 'hand']].drop_duplicates().head(10)

,player,hand
0,Aaron Krickstein,R
8,Abedallah Shelbayh,L
10,Adam Pavlasek,R
13,Adam Walton,R
15,Adolfo Daniel Vallejo,R
16,Adrian Andreev,R
22,Adrian Garcia,R
23,Adrian Mannarino,L
93,Adrian Voinea,R
94,Adriano Panatta,R


In [9]:
player_profiles.to_csv(f'{BASE}/player_profiles.csv', index=False)